# Library & Data import

In [ ]:
import pandas as pd
import numpy as np
import random

from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
import numpy as np

import geopandas as gpd
import folium
from folium.plugins import HeatMap
import pydeck as pdk
import hashlib
import plotly.express as px


import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import DBSCAN

import json

from shapely import wkt
from rapidfuzz import process, fuzz


In [ ]:
df = pd.read_csv('..\Data\\Org_W_Duplicates_data_1_0.csv', low_memory=False)
main_df = pd.read_csv('..\Data\\311_2025_Jan_Dec.csv', low_memory=False) # The dataframe before the changes we applied in FinalTable notebook
nyc_map = gpd.read_file("..\\Data\\Modified Zip Code Tabulation Areas (MODZCTA).geojson")
nyc_streets = gpd.read_file("..\\Data\\Centerline.geojson")

In [ ]:
pd.set_option('display.max_rows', 200)  # In case rows are being cut off 

# First look at the data

In [ ]:
# Dataset shape
print("There are {} rows and {} columns in the dataset".format(df.shape[0], df.shape[1]))

In [ ]:
# Quick look into the first 5 rows of the dataset
df.head() 

In [ ]:
# See each column and its data type
df.info()

<div class="valid-values">
    <h2>Valid Values</h2>
    <ul>
        <li><strong>city:</strong> Cities in the New York metropolitan area</li>
        <li><strong>Incident Zip:</strong> Zip code of incident location <em>(Dtype is str but represents a number - will be converted later)</em></li>
        <li><strong>Street Name:</strong> Street name of incident location <em></em></li>
        <li><strong>Agency:</strong> Service responsible for the complaint</li>
        <li><strong>Location Type:</strong> (Street, Sidewalk, Residential building...)</li>
        <li><strong>Status:</strong> (Closed, Open, Processing...)</li>
        <li><strong>Location:</strong> Coordinates of the complaint</li>
        <li><strong>Resolution Time:</strong> Time it took to resolve the issue</li>
        <li><strong>Complaint Type:</strong> ("Housing & Building Maintenance", "Vehicles & Parking"...)</li>
    </ul>
</div>

In [ ]:
# Lets see some of the problems people complain about
print(df["Complaint_Type"].unique())
print("=================================================")
df["Complaint_Type"].value_counts()

# Duplicates 

During data remodeling, we have found out that there is a column called "Additional Details" that causes us to have duplicates. The problem with the column is that for the same complaint, we got more rows, but why?  because each row had a different additional detail. In our clustering problem we want to focus on just the complaint type without more details so we decided to remove any duplicates that appear in our data.

In [ ]:
# Amount of duplicates
print(df.duplicated().sum())

In [ ]:
# The duplicate rows
df[df.duplicated(keep=False)]


In [ ]:
# Get the index of duplicated rows in df
dup_index = df[df.duplicated()].index
# Use that index to display the full rows from main_df
main_df.loc[dup_index]

Let's focus on a specific duplicate with id 170 and 172

In [ ]:
main_df.loc[[170,172]]

As we can see, the rows are almost identical because its the same complaint: "Plumbing" but why did it create 2 rows? Because of "Additional Details" column! We have got 2 different details for the same problem. As we said in the beginning we will remove all duplicates. ( 363809 rows which is 10% of the data)

In [ ]:
# Dropping duplicates
df = df.drop_duplicates(keep=False)
# Validating duplicates removel
print(df.duplicated().sum())

# Missing Values

In [ ]:
# Missing Values Distribution
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=True)
missing_pct = (missing / len(df) * 100).round(2)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#FF6B6B' if pct > 10 else '#FFA94D' if pct > 5 else '#74C0FC'
          for pct in missing_pct]
bars = ax.barh(missing.index, missing_pct, color=colors, edgecolor='white',
               linewidth=0.8, height=0.6)
# Annotate bars
for bar, count, pct in zip(bars, missing, missing_pct):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{pct}%  ({count:,})', va='center', ha='left',
            fontsize=10, color='#E0E0E0')
ax.set_facecolor('#1E1E2E')
fig.patch.set_facecolor('#1E1E2E')
ax.tick_params(colors='#E0E0E0', labelsize=10)
ax.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.0f%%'))
ax.set_xlabel('% of Missing Values', color='#A0A0B0', fontsize=11)
ax.set_title('Missing Values per Column', color='white', fontsize=14, fontweight='bold', pad=15)

# Legend
ax.legend( loc='lower right', facecolor='#2A2A3E',
          edgecolor='none', labelcolor='#E0E0E0', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Number of missing value for each feature.
df.isnull().sum()

In [ ]:
# Percentage of missing values for each feature.
df.isnull().mean()*100

This is the data which is Null/None/NaN... But what if the missing data was written as Unknown or Unavailable? Let's check the most common encodings.

In [ ]:
# Replace common text based fake nulls with actual NaN val
# List of common fake nulls
fake_nulls = [
    "Unspecified", "UNKNOWN", "Unknown", "unknown", 
    "Not Applicable", "N/A", "NA", "n/a", "na", 
    "None", "none", "TBD", "Other",
    "", " ", "  ", "-", ".",
    "-1", "-999", "99999" 
]

# Apply the replacement across the entire dataframe
df.replace(fake_nulls, np.nan, inplace=True)

# Now check the true damage!
print(df.isnull().sum())

We can see that some null values were added to "Location Type". Other than that, no other common fake nulls were found - Great success 

As we can see, most of the columns have missing values. Let's see what to do with them.

<div class="missing-data-summary">
    <h3 style="margin-bottom: 20px;">Reasons for missing values in each column:</h3>
    <ul style="line-height: 1.6;">
        <li style="margin-bottom: 15px;">
            <strong>City:</strong> Incomplete Complaint Submissions, &nbsp;&nbsp;&nbsp; Data Entry Errors, &nbsp;&nbsp;&nbsp; Jurisdiction Edge Cases.
        </li>
        <li style="margin-bottom: 15px;">
            <strong>Street name:</strong> Unrecognized or Undefined Street, &nbsp;&nbsp;&nbsp; Data error, Jurisdiction Edge Cases.
        </li>
        <li style="margin-bottom: 15px;">
            <strong>Incident Zip:</strong> Complaints Filed Without a Specific Address, &nbsp;&nbsp;&nbsp; Homeless/Transient Complaints.
        </li>
        <li style="margin-bottom: 15px;">
            <strong>Location:</strong> Privacy Redaction, &nbsp;&nbsp;&nbsp; No Physical Address Exists, &nbsp;&nbsp;&nbsp; Failed GPS Capture.
        </li>
        <li style="margin-bottom: 15px;">
            <strong>Status:</strong> Transferred Between Agencies, &nbsp;&nbsp;&nbsp; Legacy/Migrated Data.
        </li>
        <li style="margin-bottom: 15px;">
            <strong>Closed Date:</strong> Complaint Is Still Open, &nbsp;&nbsp;&nbsp; Complaint Is Pending or In Progress.
        </li>
    </ul>
</div>

## City

There are 155,630 missing values.

We can try to determine the City using the Zip ( as long as the zip is not missing as well).

In [ ]:
# Print the rows with missing values
df[df['City'].isna()].head()


In [ ]:
# Filter rows where City is missing
missing_city = df[df['City'].isna()]

# Check if "Incident Zip" is not null for those rows
all_have_location = missing_city['Location'].notna().all()

print(f"Rows with missing City: {len(missing_city)}")
print(f"Of those, rows WITH Location: {missing_city['Incident Zip'].notna().sum()}")
print(f"Of those, rows WITHOUT Location: {missing_city['Incident Zip'].isna().sum()}")


In [ ]:
# Loading the NYC Borough Boundaries map from a stable GitHub mirror
url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/new-york-city-boroughs.geojson"
nyc_boundaries = gpd.read_file(url)

# Renaming for easier use
nyc_boundaries = nyc_boundaries[['name', 'geometry']].rename(columns={'name': 'boro_name'})

# Removing missing values 
missing_city_df = df[df['City'].isna()].dropna(subset=['Location']).copy()

# Turning missing city data into a GeoDataFrame
points_gdf = gpd.GeoDataFrame(
    missing_city_df, 
    geometry=gpd.GeoSeries.from_wkt(missing_city_df['Location']),
    crs="EPSG:4326" # Standard coordinate system (WGS84)
)

# The boundry map must use the same standard coordinate system
nyc_boundaries = nyc_boundaries.to_crs("EPSG:4326")

# This checks which polygon (borough) every point falls within
joined_gdf = gpd.sjoin(points_gdf, nyc_boundaries, how="left", predicate="within")

# Updating the original dataframe with the newly found Borough/City names
df.loc[joined_gdf.index, 'City'] = joined_gdf['boro_name'].str.title()

print(f"Remaining missing: {df['City'].isna().sum()}")

We can't really fill the rest of the missing values of City. The easiest way to deal with this problem is to remove the missing values lines. 30k rows are approximatly  ~1% of the total rows in the dataset so it won't affect it much.

In [ ]:
df = df.dropna(subset=['City'])

print(f"Remaining rows: {len(df)}")

In [ ]:
print(df.isnull().sum())

## Incident Zip

There are 29,771 missing values

Unlike City, we can't just fill the missing values by knowing in which city the complaint was created

In [ ]:
# Print the rows with missing values
df[df['Incident Zip'].isna()].head()


We can't really fill the missing values of Incident Zip. The easiest way to deal with this problem is to remove the missing values lines. 12k rows are approximatly  ~0.3% of the total rows in the dataset so it won't affect it much.

In [ ]:
df = df.dropna(subset=['Incident Zip'])

print(f"Remaining rows: {len(df)}")

In [ ]:
df.isnull().sum()

## Location

There are 19,970 missing values

There is no way that we are able to find the exact Location of the complaint and 20k rows are only ~1% of the data so we can remove them without affecting the data.

In [ ]:
df = df.dropna(subset=['Location'])

print(f"Remaining rows: {len(df)}")

In [ ]:
df.isnull().sum()

## Status

Only 169 rows are missing, no real value in researching and filling those missing values...

In [ ]:
df = df.dropna(subset=['Status'])

print(f"Remaining rows: {len(df)}")

In [ ]:
df.isnull().sum()

## Closed Date

There are 67,220 missing values

If we solve the closed date missing values issue , then we would also be able to calculate the Resoultion Time!

In [ ]:
# Print the rows with missing values
df[df['Closed Date'].isna()].head()


In [ ]:
# Create a flag: 1 = Closed Date is missing, 0 = not missing
df['closed_date_missing'] = df['Closed Date'].isna().astype(int)

# Group by City and calculate the missing rate
city_missing = df.groupby('City')['closed_date_missing'].agg(['sum', 'count', 'mean'])
city_missing.columns = ['Missing Count', 'Total Complaints', 'Missing Rate']
city_missing = city_missing.sort_values('Missing Rate', ascending=False)
df = df.drop('closed_date_missing', axis=1)

# Only show cities with enough data points to be meaningful
top_cities = city_missing[city_missing['Total Complaints'] > 50]
top_cities['Missing Rate'].plot(kind='bar', figsize=(14, 5), title='Missing Closed Date Rate by City')
plt.ylabel('Missing Rate')
plt.tight_layout()
plt.show()

We can see that the percentage of missing values per total complaints is the highest in "Howard Beach" with 25%. Indicating some kind of correlation.

In [ ]:
# We have missing values because the complaint is probably still not solved. Lets check the Status column 
df[df['Closed Date'].isna()]["Status"].value_counts()

Most complaints are stll in progress thus there is no Closed Date reported.

In [ ]:
df = df.dropna(subset=['Closed Date'])

print(f"Remaining rows: {len(df)}")

## Street Name

There are 92k missing values left. Lets check if the missing value in the street column correlates with the location of the complaint.

In [ ]:
df.isnull().sum()

In [ ]:
# Filter rows where Street Name is missing
missing_street = df[df['Street Name'].isna()]

# Let's check both Location and Zip Code availability
print(f"Rows with missing Street Name: {len(missing_street)}")
print(f"Of those, rows WITH Location data: {missing_street['Location'].notna().sum()}")


Well good result, we can fill the data using the location of the report.

In [ ]:
# Check if we added new streets
df['Street Name'].nunique()

In [ ]:
# We will use it later to fill missing values in street name column
nyc_streets["stname_label"].unique()

In [ ]:
# Isolate the rows that need fixing
missing_street_mask = df['Street Name'].isna() & df['Location'].notna()
df_missing = df[missing_street_mask].copy()

# If the data has a comma, like POINT(-73.9, 40.7), we remove it first so we can work with wkt
df_missing['Location'] = df_missing['Location'].str.replace(',', '', regex=False)

# Turn the WKT text directly into geometry  ( We don't remove Location since geometry objects can't be exporeted as csv)
df_missing['geometry'] = gpd.GeoSeries.from_wkt(df_missing['Location'])

# Convert into a GeoDataFrame 
missing_points = gpd.GeoDataFrame(
    df_missing, 
    geometry='geometry',
    crs="EPSG:4326" # Standard GPS coordinates
)

print(f"Rows: {len(missing_points)} | CRS: {missing_points.crs}")
missing_points.head()

In [ ]:
# Lets take a look at nyc_streets
nyc_streets.head()

In [ ]:
# Project to NY State Plane (feet) to apply an accurate buffer
streets_projected = nyc_streets.to_crs(epsg=2263) # 2263 local epsg for nyc

# Buffer by 40 feet to create a polygon representing the road and sidewalk
streets_projected['geometry'] = streets_projected.geometry.buffer(40)

# Project the missing points to match the streets
missing_points_projected = missing_points.to_crs(epsg=2263)

# Perform the Spatial Join
matched_data = gpd.sjoin(
    missing_points_projected, 
    streets_projected[['full_street_name', 'geometry']], 
    how="left", 
    predicate="within"
)

matched_data = matched_data[~matched_data.index.duplicated(keep='first')]

# Patch the original DataFrame
df.loc[missing_street_mask, 'Street Name'] = matched_data['full_street_name']

print(f"Recovered street names for {matched_data['full_street_name'].notna().sum()} rows. Out of 92324 needed")

In [ ]:
# Missing values for the Joined df
matched_data.isnull().sum()

In [ ]:
# Joined df
matched_data.head()

In [ ]:
# Post join
df.isnull().sum()

We have 57 streets that we couldnt find maybe because of invalid location. We will just drop them.

In [ ]:
df = df.dropna(subset=['Street Name'])
df.isnull().sum()

# Outliers

## Resolution Time

In our data the only numerical column is the "Resoultion Time" column. We will use the describe method on it.

In [ ]:
df["Resolution Time"].describe()

According to the describe of the column above, we have some outliers. Let's print the rows and see whats the problem.

In [ ]:
df[df["Resolution Time"] < 0]

As we expected the problem was with the creation date and the closing date, one possible explaination for it is a human input error... To fix this issue we will assume the dates were swapped so the best way to is to just multiply the negative resolution time by -1 and swap the dates.

In [ ]:
# Identify rows where Resolution Time is negative (dates were swapped)
mask = df["Resolution Time"] < 0

# Swap Created Date and Closed Date for those rows
df.loc[mask, ["Created Date", "Closed Date"]] = (
    df.loc[mask, ["Closed Date", "Created Date"]].values
)

# Flip the negative Resolution Time to positive
df.loc[mask, "Resolution Time"] *= -1


In [ ]:
df["Resolution Time"].describe()

## Street Name

### Normalizing data

Some streets may be written in a different format or have prefix or suffix spaces in the string e.g: " 4 AVENUE" or "4 avenue" or "____4 AVENUE" 

In [ ]:
# To fix the issue , we will normalize the data ( and later the nyc map itself ) and convert it to the same format
# Convert everything to UPPERCASE
df['Street Name'] = df['Street Name'].str.upper()

# Strip hidden spaces from the very beginning or end of the text
df['Street Name'] = df['Street Name'].str.strip()

# Crush multiple internal spaces into a single space (from our previous fix)
df['Street Name'] = df['Street Name'].str.replace(r'\s+', ' ', regex=True)

#===================================================
# Lets validate the number of matches

# Get a unique list of all official NYC street names
# Using the geodf from "missing values" section
official_streets = set(streets_projected['full_street_name'].dropna().unique())

# Get a unique list of the street names currently in the dataset
my_streets = set(df['Street Name'].dropna().unique())

# Find outliers difference (Streets in our data that do NOT exist in the official data)
unmatched_streets = my_streets - official_streets

# Print the results
print(f"Total unique streets in our dataset: {len(my_streets)}")
print(f"Streets that perfectly match the official map: {len(my_streets & official_streets)}")
print(f"Streets that STILL don't match: {len(unmatched_streets)}")

We got a lot of mismatches, one reason could be that the data in the nyc geodf itself is NOT normalized like we did on the data, thus we will normalize it exactly like the code above.

In [ ]:
# Uppercase the official streets
streets_projected['full_street_name'] = streets_projected['full_street_name'].str.upper()

# Crush multiple spaces into a single space in the official data
streets_projected['full_street_name'] = streets_projected['full_street_name'].str.replace(r'\s+', ' ', regex=True)

# Strip any trailing/leading spaces
streets_projected['full_street_name'] = streets_projected['full_street_name'].str.strip()


#===================================================
# Lets validate the number of matches

# Recreate the official set now that its clean
official_streets = set(streets_projected['full_street_name'].dropna().unique())

# Find the difference ( my_street from the previous code )
unmatched_streets = my_streets - official_streets

print(f"Total unique streets in your dataset: {len(my_streets)}")
print(f"Streets that perfectly match the official map: {len(my_streets & official_streets)}")
print(f"Streets that STILL don't match: {len(unmatched_streets)}")

In [ ]:
# Let's look at the first 15 weird ones to see what's wrong
print("\nSample of unmatched streets:")
print(random.sample(list(unmatched_streets), 15))

We see in the sample of 15 unmatched streets that are some streets that use the shortened version of the word like ST which is STREET. Moreover, some streets have numbers written in the text format ( Five , Three ... ) and not in the normalized int format ( "5", "3", ... ) .

We will have to fix this manually in the codes below.

### Removing short words & Using fuzz library to match almost identical words

In [ ]:
# ============================================================
# STEP 1 — Normalize both datasets (whitespace, case, punctuation)
# ============================================================

def normalize(series):
    return (series
        .str.strip()
        .str.upper()
        .str.replace(r'\s+', ' ', regex=True)      # collapse multiple spaces
        .str.replace(r'[^\w\s]', '', regex=True))   # remove punctuation

df['Street Name'] = normalize(df['Street Name'])
streets_projected['full_street_name'] = normalize(streets_projected['full_street_name'])


# ============================================================
# STEP 2 — Expand abbreviations in both datasets
# ============================================================

nyc_expansions = {
    r'\bST\b':   'STREET',
    r'\bAVE\b':  'AVENUE',
    r'\bBLVD\b': 'BOULEVARD',
    r'\bRD\b':   'ROAD',
    r'\bPL\b':   'PLACE',
    r'\bPKWY\b': 'PARKWAY',
    r'\bW\b':    'WEST',
    r'\bE\b':    'EAST',
    r'\bN\b':    'NORTH',
    r'\bS\b':    'SOUTH',
    r'\bDR\b':   'DRIVE',
    r'\bCT\b':   'COURT',
    r'\bLN\b':   'LANE',
    r'\bTPKE\b': 'TURNPIKE',
    r'\bEXPY\b': 'EXPRESSWAY',
    r'\bEXPWY\b': 'EXPRESSWAY',
    r'\bSQ\b':   'SQUARE',
    r'\bTER\b':  'TERRACE',

    # Extra abbreviations
    r'\bHWY\b':  'HIGHWAY',
    r'\bCRES\b': 'CRESCENT',
    r'\bALY\b':  'ALLEY',
    r'\bCIR\b':  'CIRCLE',
    r'\bFWY\b':  'FREEWAY',

    # Popular road names
    r'\bFDR\b':   'FRANKLIN D ROOSEVELT',
    r'\bBQE\b':   'BROOKLYN QUEENS EXPRESSWAY',
    r'\bLIE\b':   'LONG ISLAND EXPRESSWAY',
    r'\bGCP\b':   'GRAND CENTRAL PARKWAY',
    r'\bRSD\b':   'RIVERSIDE DRIVE',
    r'\bHRD\b':   'HARLEM RIVER DRIVE',
    r'\bCPW\b':   'CENTRAL PARK WEST',
    r'\bPPW\b':   'PROSPECT PARK WEST',
    r'\bACP\b':   'ADAM CLAYTON POWELL',
    r'\bBWAY\b':  'BROADWAY',
    r'\bMLK\b':   'MARTIN LUTHER KING',
}
for pattern, full_word in nyc_expansions.items():
    df['Street Name'] = df['Street Name'].str.replace(pattern, full_word, regex=True)
    streets_projected['full_street_name'] = streets_projected['full_street_name'].str.replace(pattern, full_word, regex=True)


# ============================================================
# STEP 3 — Convert written-out numbers to digits (df only)
# ============================================================

written_number_fixes = {
    r'\bFIRST\b':     '1',
    r'\bSECOND\b':    '2',
    r'\bTHIRD\b':     '3',
    r'\bFOURTH\b':    '4',
    r'\bFIFTH\b':     '5',
    r'\bSIXTH\b':     '6',
    r'\bSEVENTH\b':   '7',
    r'\bEIGHTH\b':    '8',
    r'\bNINTH\b':     '9',
    r'\bTENTH\b':     '10',
    r'\bELEVENTH\b':  '11',
    r'\bTWELFTH\b':   '12',
    r'\bTHIRTEENTH\b':'13',
    r'\bFOURTEENTH\b':'14',
    r'\bFIFTEENTH\b': '15',
}

df['Street Name'] = df['Street Name'].replace(written_number_fixes, regex=True)


# ============================================================
# STEP 4 — Strip ordinal suffixes from both datasets
#           1ST → 1, 2ND → 2, 3RD → 3, 42ND → 42, etc.
# ============================================================

ordinal_fixes = {
    r'\b(\d+)ST\b': r'\1',
    r'\b(\d+)ND\b': r'\1',
    r'\b(\d+)RD\b': r'\1',
    r'\b(\d+)TH\b': r'\1',
}

for pattern, replacement in ordinal_fixes.items():
    df['Street Name'] = df['Street Name'].str.replace(pattern, replacement, regex=True)
    streets_projected['full_street_name'] = streets_projected['full_street_name'].str.replace(pattern, replacement, regex=True)


# ============================================================
# STEP 5 — Fuzzy match whatever is still unmatched
# ============================================================

official_streets = set(streets_projected['full_street_name'].dropna().unique())
my_streets       = set(df['Street Name'].dropna().unique())
unmatched_streets = my_streets - official_streets

print(f"Unmatched streets before fuzzy: {len(unmatched_streets)}")

def fuzzy_match(street_name, official_set, threshold=88):
    result = process.extractOne(
        street_name,
        official_set,
        scorer=fuzz.token_sort_ratio   # handles word-order differences too
    )
    if result and result[1] >= threshold:
        return result[0]
    return None

fuzzy_map = {}
for street in unmatched_streets:
    match = fuzzy_match(street, official_streets)
    if match:
        fuzzy_map[street] = match

df['Street Name'] = df['Street Name'].replace(fuzzy_map)
print(f"Fuzzy matched an additional {len(fuzzy_map)} streets")


# ============================================================
# STEP 6 — Final validation
# ============================================================

official_streets = set(streets_projected['full_street_name'].dropna().unique())
my_streets       = set(df['Street Name'].dropna().unique())
unmatched_streets = my_streets - official_streets

print("\n=== FINAL MATCH RESULTS ===")
print(f"Total unique streets in your dataset : {len(my_streets)}")
print(f"Streets that perfectly match         : {len(my_streets & official_streets)}")
print(f"Streets still unmatched              : {len(unmatched_streets)}")

df_has_street = df.dropna(subset=['Street Name'])
matched_rows  = df_has_street[df_has_street['Street Name'].isin(official_streets)]
match_pct     = (len(matched_rows) / len(df_has_street)) * 100

print(f"\nTotal complaint rows matched    : {len(matched_rows):,} ({match_pct:.1f}%)")
print(f"Total complaint rows left behind: {len(df_has_street) - len(matched_rows):,}")

In [ ]:
streets_projected["full_street_name"]

In [ ]:
streets_projected.head()

### Manual fixing

In [ ]:
# Print the unmatched streets for manual overview
print(list(unmatched_streets))

After printing the unmatched "streets" we can see that some of the names are not actually streets like parks or statues so we can safely remove them.

Non_street_keywords contains keywords that we manually checked that do NOT represent streets.

In [ ]:
# Keywords that indicate a real street
street_keywords = [
    'STREET', 'AVENUE', 'BOULEVARD', 'ROAD', 'DRIVE', 'LANE', 
    'PLACE', 'PARKWAY', 'HIGHWAY', 'EXPRESSWAY', 'TURNPIKE', 
    'CIRCLE', 'TERRACE', 'COURT', 'CRESCENT', 'ALLEY', 'WAY',
    'CONCOURSE', 'MALL', 'MALLS', 'PLAZA', 'PATH', 'WALK'
]

# Keywords that are clearly NOT streets
non_street_keywords = [
    'PARK', 'PLAYGROUND', 'PLGD', 'PIER', 'HOSPITAL', 'SCHOOL',
    'COLLEGE', 'UNIVERSITY', 'AIRPORT', 'BRIDGE', 'CEMETERY',
    'MUSEUM', 'STADIUM', 'GARDEN', 'POOL', 'FIELD', 'BEACH',
    'ARENA', 'CENTER', 'TERMINAL', 'LIBRARY', 'THEATER', 'THEATRE',
    'ISLAND', 'BAY', 'CREEK', 'POND', 'LAKE', 'RESERVOIR',
    'HS', 'ARPT', 'JFK', 'LGA', 'TOWER', 'BUILDING', 'HALL',
    'HOUSES', 'HOUSES', 'BASIN', 'COVE', 'HEIGHTS', 'SQUARE',
    'TRIANGLE', 'CIRCLE', 'ZONE', 'ZOO', 'BOTANICAL', 'PCT',
    'AQUEDUCT', 'YESHIVA', 'MARINA', 'PARKING', 'PARKS', 'FERRY'
]

def classify_unmatched(name):
    words = set(name.split())
    has_street_word = any(kw in words for kw in street_keywords)
    has_non_street_word = any(kw in words for kw in non_street_keywords)
    
    if has_street_word and not has_non_street_word:
        return 'possible_street'
    elif has_non_street_word:
        return 'not_a_street'
    else:
        return 'unclear'  # manual review

classifications = {name: classify_unmatched(name) for name in unmatched_streets}

possible_streets = [n for n, c in classifications.items() if c == 'possible_street']
not_streets      = [n for n, c in classifications.items() if c == 'not_a_street']
unclear          = [n for n, c in classifications.items() if c == 'unclear']

print(f"Possible streets to retry matching : {len(possible_streets)}")
print(f"Non-streets safe to drop           : {len(not_streets)}")
print(f"Unclear (needs manual review)      : {len(unclear)}")

# Print unclear ones — these are the only ones worth eyeballing
print("\n--- Unclear (manual review) ---")
for s in sorted(unclear):
    print(s)

# Drop rows where the street name is confirmed non-street
df = df[~df['Street Name'].isin(not_streets)]

In [ ]:
print(len(unclear))
print(unclear)

Problematic names we encountered: 
Miss-spelled (e.g: 'BENNET REST' -> 'BENNET AVENUE')
Missing suffix (e.g: 'ARTHUR KILL' -> actually 'ARTHUR KILL ROAD')
Flipped names (e.g: 'ARDEN WOODS' -> 'WOODS OF ARDEN STREET')


We will fix each problem since each street we miss, is a street that won't be printed on the final map making it look empty.

In [ ]:
# ============================================================
# MANUAL STREET NAME FIXES ( Used LLM to map the wrong street names to the real ones)
# ============================================================

street_mapping = {
    "BOONE SLOPE": "BOONE AVENUE",
    "OTTAVIO PROMENADE": "OTTAVIO PROMENADE",
    "NORTH SHORE ESPLANADE": "NORTH SHORE ESPLANADE",
    "EAST RIVER ESPLANADE": "EAST RIVER ESPLANADE",
    "WALTON SLOPE": "WALTON AVENUE",
    "JEROME SLOPE": "JEROME AVENUE",
    "JERRICO TNPK": "JERICHO TURNPIKE",
    "EINSTEIN LOOP": "EINSTEIN LOOP",
    "STAPLETON ESPLANADE": "STAPLETON ESPLANADE",
    "BENNETT REST": "BENNETT AVENUE",
    "EAGLE SLOPE": "EAGLE AVENUE",
    "NEW ENGLAND THRU": "NEW ENGLAND THRUWAY",
    "MAYBERRY PROMENADE": "MAYBERRY PROMENADE",
    "EDERLE PROMENADE": "EDERLE PROMENADE",
    "KINGSHIGWY": "KINGS HIGHWAY",
    "NORTHERN BL SR SOUTH": "NORTHERN BOULEVARD SOUTH SERVICE ROAD",
    "RIEGELMANN BOARDWALK": "RIEGELMANN BOARDWALK",
    "SHORE PW": "SHORE PARKWAY",
    "SOUTHBEACH BOARDWALK": "SOUTH BEACH BOARDWALK",
    "EAST 183 PED BR OV METRO NORTH HRLM": "EAST 183RD PEDESTRIAN BRIDGE OVER METRO NORTH HARLEM",
    "BUSHMAN STEPS": "BUSHMAN STEPS",
    "QNS MIDTOWN TUNNEL": "QUEENS MIDTOWN TUNNEL",
    "NORTHSHORE ESPLANADE": "NORTH SHORE ESPLANADE",
    "UNION TNPK": "UNION TURNPIKE",
    "LA 65 DE INFANTERIA": "65TH INFANTRY BOULEVARD",
    "INDUSTRIAL LOOP WEST": "INDUSTRIAL LOOP WEST",
    "RIEGLEMANN BOARDWALK": "RIEGELMANN BOARDWALK",
    "CPS": "CENTRAL PARK SOUTH",
    "CON VILLAGE WEST": "CONCOURSE VILLAGE WEST",
    "OCEAN DRIVEWAY": "OCEAN DRIVEWAY",
    "PARKLANE SOUTH": "PARK LANE SOUTH",
    "ASTORIA BL NORTH": "ASTORIA BOULEVARD NORTH",
    "NEW ENG THRWY": "NEW ENGLAND THRUWAY",
    "DIANAS TRAIL": "DIANAS TRAIL",
    "SEMINARY ROW": "SEMINARY ROW",
    "OCEANDRIVEWAY": "OCEAN DRIVEWAY",
    "JUNIPER BL SOUTH": "JUNIPER BOULEVARD SOUTH",
    "HALFMOON ISLE": "HALFMOON ISLE",
    "MIDTOWN TUNNEL": "QUEENS MIDTOWN TUNNEL",
    "CANYON OF HEROES": "BROADWAY",
    "OCEAN PROMENADE": "OCEAN PROMENADE",
    "RANDALLS IS QUEENS PEDESTRIAN RP": "RANDALLS ISLAND QUEENS PEDESTRIAN RAMP",
    "BKLYN BATTERY TUNNEL": "BROOKLYN BATTERY TUNNEL",
    "WEST SIDE HW": "WEST SIDE HIGHWAY",
    "HAVEN ESPLANADE": "HAVEN ESPLANADE",
    "RIEGELMAN BOARDWALK": "RIEGELMANN BOARDWALK",
    "I 278": "INTERSTATE 278",
    "NORTHERN BL EAST": "NORTHERN BOULEVARD EAST",
    "I 678": "INTERSTATE 678",
    "LONG ISL MEWS": "LONG ISLAND MEWS",
    "LIPPMAN ARCADE": "LIPPMAN ARCADE",
    "GRAND SLOPE": "GRAND CONCOURSE",
    "EAST RIV": "EAST RIVER DRIVE",
    "LAKEVIEW BL EAST": "LAKEVIEW BOULEVARD EAST",
    "BATTERY TUNNEL": "BROOKLYN BATTERY TUNNEL",
    "GRANDCONCOURSE": "GRAND CONCOURSE",
    "THE HIGH LINE": "THE HIGH LINE",
    "FT GEORGE HILL": "FORT GEORGE HILL",
    "CATHERINE SCOTT PROMENADE": "CATHERINE SCOTT PROMENADE"
}

# Apply
df['Street Name'] = df['Street Name'].replace(street_mapping)

# Verify each fix actually lands in official streets
print("=== Manual Fix Verification ===")
matched   = 0
unmatched = 0

for original, fixed in street_mapping.items():
    if fixed in official_streets:
        print(f"  ✓  '{original}' → '{fixed}'")
        matched += 1
    else:
        print(f"  ✗  '{original}' → '{fixed}'  ← NOT IN OFFICIAL STREETS")
        unmatched += 1

print(f"\n{matched} fixed, {unmatched} still not matching official streets")



### Missing suffix

In [ ]:
# Missing suffix fix
suffixes = [
    'STREET', 'AVENUE', 'BOULEVARD', 'ROAD', 'DRIVE', 'LANE',
    'PLACE', 'PARKWAY', 'HIGHWAY', 'EXPRESSWAY', 'COURT',
    'TERRACE', 'CIRCLE', 'CRESCENT', 'WAY', 'ALLEY', 'PATH'
]

suffix_map = {}

for name in unclear:
    for suffix in suffixes:
        candidate = f"{name} {suffix}"
        if candidate in official_streets:
            suffix_map[name] = candidate
            break  # stop at first match

print(f"Recovered {len(suffix_map)} streets by appending suffix:")
for original, fixed in suffix_map.items():
    print(f"  '{original}' → '{fixed}'")

# Apply the fix to df
df['Street Name'] = df['Street Name'].replace(suffix_map)

### Running the first main code again

In [ ]:
# ============================================================
# STEP 1 — Normalize both datasets (whitespace, case, punctuation)
# ============================================================

def normalize(series):
    return (series
        .str.strip()
        .str.upper()
        .str.replace(r'\s+', ' ', regex=True)      # collapse multiple spaces
        .str.replace(r'[^\w\s]', '', regex=True))   # remove punctuation

df['Street Name'] = normalize(df['Street Name'])
streets_projected['full_street_name'] = normalize(streets_projected['full_street_name'])


# ============================================================
# STEP 2 — Expand abbreviations in both datasets
# ============================================================

nyc_expansions = {
    r'\bST\b':   'STREET',
    r'\bAVE\b':  'AVENUE',
    r'\bBLVD\b': 'BOULEVARD',
    r'\bRD\b':   'ROAD',
    r'\bPL\b':   'PLACE',
    r'\bPKWY\b': 'PARKWAY',
    r'\bW\b':    'WEST',
    r'\bE\b':    'EAST',
    r'\bN\b':    'NORTH',
    r'\bS\b':    'SOUTH',
    r'\bDR\b':   'DRIVE',
    r'\bCT\b':   'COURT',
    r'\bLN\b':   'LANE',
    r'\bTPKE\b': 'TURNPIKE',
    r'\bEXPY\b': 'EXPRESSWAY',
    r'\bEXPWY\b': 'EXPRESSWAY',
    r'\bSQ\b':   'SQUARE',
    r'\bTER\b':  'TERRACE',

    # Extra abbreviations
    r'\bHWY\b':  'HIGHWAY',
    r'\bCRES\b': 'CRESCENT',
    r'\bALY\b':  'ALLEY',
    r'\bCIR\b':  'CIRCLE',
    r'\bFWY\b':  'FREEWAY',

    # Popular road names
    r'\bFDR\b':   'FRANKLIN D ROOSEVELT',
    r'\bBQE\b':   'BROOKLYN QUEENS EXPRESSWAY',
    r'\bLIE\b':   'LONG ISLAND EXPRESSWAY',
    r'\bGCP\b':   'GRAND CENTRAL PARKWAY',
    r'\bRSD\b':   'RIVERSIDE DRIVE',
    r'\bHRD\b':   'HARLEM RIVER DRIVE',
    r'\bCPW\b':   'CENTRAL PARK WEST',
    r'\bPPW\b':   'PROSPECT PARK WEST',
    r'\bACP\b':   'ADAM CLAYTON POWELL',
    r'\bBWAY\b':  'BROADWAY',
    r'\bMLK\b':   'MARTIN LUTHER KING',
}
for pattern, full_word in nyc_expansions.items():
    df['Street Name'] = df['Street Name'].str.replace(pattern, full_word, regex=True)
    streets_projected['full_street_name'] = streets_projected['full_street_name'].str.replace(pattern, full_word, regex=True)


# ============================================================
# STEP 3 — Convert written-out numbers to digits (df only)
# ============================================================

written_number_fixes = {
    r'\bFIRST\b':     '1',
    r'\bSECOND\b':    '2',
    r'\bTHIRD\b':     '3',
    r'\bFOURTH\b':    '4',
    r'\bFIFTH\b':     '5',
    r'\bSIXTH\b':     '6',
    r'\bSEVENTH\b':   '7',
    r'\bEIGHTH\b':    '8',
    r'\bNINTH\b':     '9',
    r'\bTENTH\b':     '10',
    r'\bELEVENTH\b':  '11',
    r'\bTWELFTH\b':   '12',
    r'\bTHIRTEENTH\b':'13',
    r'\bFOURTEENTH\b':'14',
    r'\bFIFTEENTH\b': '15',
}

df['Street Name'] = df['Street Name'].replace(written_number_fixes, regex=True)


# ============================================================
# STEP 4 — Strip ordinal suffixes from both datasets
#           1ST → 1, 2ND → 2, 3RD → 3, 42ND → 42, etc.
# ============================================================

ordinal_fixes = {
    r'\b(\d+)ST\b': r'\1',
    r'\b(\d+)ND\b': r'\1',
    r'\b(\d+)RD\b': r'\1',
    r'\b(\d+)TH\b': r'\1',
}

for pattern, replacement in ordinal_fixes.items():
    df['Street Name'] = df['Street Name'].str.replace(pattern, replacement, regex=True)
    streets_projected['full_street_name'] = streets_projected['full_street_name'].str.replace(pattern, replacement, regex=True)


# ============================================================
# STEP 5 — Fuzzy match whatever is still unmatched
# ============================================================

official_streets = set(streets_projected['full_street_name'].dropna().unique())
my_streets       = set(df['Street Name'].dropna().unique())
unmatched_streets = my_streets - official_streets

print(f"Unmatched streets before fuzzy: {len(unmatched_streets)}")

def fuzzy_match(street_name, official_set, threshold=88):
    result = process.extractOne(
        street_name,
        official_set,
        scorer=fuzz.token_sort_ratio   # handles word-order differences too
    )
    if result and result[1] >= threshold:
        return result[0]
    return None

fuzzy_map = {}
for street in unmatched_streets:
    match = fuzzy_match(street, official_streets)
    if match:
        fuzzy_map[street] = match

df['Street Name'] = df['Street Name'].replace(fuzzy_map)
print(f"Fuzzy matched an additional {len(fuzzy_map)} streets")


# ============================================================
# STEP 6 — Final validation
# ============================================================

official_streets = set(streets_projected['full_street_name'].dropna().unique())
my_streets       = set(df['Street Name'].dropna().unique())
unmatched_streets = my_streets - official_streets

print("\n=== FINAL MATCH RESULTS ===")
print(f"Total unique streets in our dataset : {len(my_streets)}")
print(f"Streets that perfectly match         : {len(my_streets & official_streets)}")
print(f"Streets still unmatched              : {len(unmatched_streets)}")

df_has_street = df.dropna(subset=['Street Name'])
matched_rows  = df_has_street[df_has_street['Street Name'].isin(official_streets)]
match_pct     = (len(matched_rows) / len(df_has_street)) * 100

print(f"\nTotal complaint rows matched    : {len(matched_rows):,} ({match_pct:.1f}%)")
print(f"Total complaint rows left behind: {len(df_has_street) - len(matched_rows):,}")

We have got a really good result, we know the accurate and real street name for almost 99% of the data. The problem is that we have the other 1% and we can't manually fix all of the streets since there is a huge amount of corrupt street names ( 1% is 35k corrupt street inputs ). We rather delete those 35k rows and operate on the other data.

In [ ]:
before = len(df)

df = df[
    df['Street Name'].isna() |
    df['Street Name'].isin(official_streets)
]

print(f"Removed {before - len(df):,} rows with unmatched street names")
print(f"Remaining rows: {len(df):,}")

### Plot

In [ ]:
# ============================================================
# STEP 1 — All streets + complaint counts
# ============================================================

street_counts = (df['Street Name']
                 .value_counts()
                 .reset_index())
street_counts.columns = ['full_street_name', 'complaint_count']

# ============================================================
# STEP 2 — Filter, dissolve, merge counts
# ============================================================

streets_to_plot = streets_projected[
    streets_projected['full_street_name'].isin(street_counts['full_street_name'])
][['full_street_name', 'geometry']].copy()

streets_to_plot = streets_to_plot.dissolve(by='full_street_name').reset_index()
streets_to_plot = streets_to_plot.merge(street_counts, on='full_street_name', how='left')

# ============================================================
# STEP 3 — Reproject to 4326
# ============================================================

streets_to_plot = streets_to_plot.to_crs(epsg=4326)

print(f"Rendering {len(streets_to_plot):,} streets...")

# ============================================================
# STEP 4 — Plot
# ============================================================

geojson = json.loads(streets_to_plot.to_json())

fig = px.choropleth_map(
    streets_to_plot,
    geojson=geojson,
    locations=streets_to_plot.index,
    color='complaint_count',
    color_continuous_scale='Reds',
    hover_data={
        'full_street_name': True,
        'complaint_count':  True,
    },
    map_style='carto-positron',
    zoom=10,
    center={"lat": 40.73, "lon": -73.93},
    opacity=0.85,
    height=750,
    title='All NYC Complaint Streets',
)

fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    coloraxis_colorbar=dict(title="Complaints"),
)

fig.show()

# Visualization

## Street

Overall streets map using the official map.

In [ ]:
# Set up a large, high resolution canvas
fig, ax = plt.subplots(1, 1, figsize=(14, 14))

# Plot only the street lines
# We use a very thin linewidth and dark color to show the grid structure
streets_projected.plot(
    ax=ax, 
    color='#2c3e50', 
    linewidth=0.2, 
    alpha=0.8
)

# Clean up the map presentation
plt.title('New York City Street Network', fontsize=18, fontweight='bold', pad=20)
plt.axis('off') # Hides the latitude/longitude axes lines
plt.tight_layout()

# Display the map
plt.show()

Overall streets map using the OUR data.

## Resulotion Time 

In [ ]:
# Compute mean and median per location type
loc_mean = df.groupby("Location Type")["Resolution Time"].mean().dropna() // 24
loc_med  = df.groupby("Location Type")["Resolution Time"].median().dropna() // 24
# Align and sort by mean
common   = loc_mean.index.intersection(loc_med.index)
loc_mean = loc_mean.loc[common].sort_values(ascending=True)
loc_med  = loc_med.loc[loc_mean.index]
# Red gradient
norm   = mcolors.Normalize(vmin=loc_mean.min(), vmax=loc_mean.max())
colors = cm.Reds([0.25 + 0.75 * norm(v) for v in loc_mean.values])
fig, ax = plt.subplots(figsize=(12, max(5, len(loc_mean) * 0.5)))
# Mean bars
bars = ax.barh(
    loc_mean.index,
    loc_mean.values,
    color=colors,
    edgecolor="white",
    linewidth=0.6,
    height=0.65,
    label="Mean"
)
# Median markers
ax.scatter(
    loc_med.values,
    loc_med.index,
    color="#2c3e50",
    marker="|",
    s=200,
    linewidths=2.5,
    zorder=5,
    label="Median"
)
# Annotate mean at end of each bar
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + loc_mean.max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.0f}d",
        va="center", ha="left", fontsize=9, color="#333333"
    )
# Overall median reference line
overall_med = loc_mean.median()
ax.axvline(overall_med, color="#4f8ef7", linewidth=1.8,
           linestyle="--", label=f"Overall Median: {overall_med:.0f}d")
ax.set_title("Resolution Time per Location Type (Mean + Median)", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Resolution Time (days)", fontsize=11)
ax.set_ylabel("Location Type", fontsize=11)
ax.set_xlim(0, loc_mean.max() * 1.18)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=10, loc="lower right")
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Compute average resolution time per city, sorted ascending (longest bar on top)
city_res = (
    df.groupby("City")["Resolution Time"]
    .mean()
    .dropna()
    .sort_values(ascending=True)//24 # Convert from Hours to Days
)

# Normalize values → [0.25, 1.0] so even the smallest bar has a visible red tint
norm = mcolors.Normalize(vmin=city_res.min(), vmax=city_res.max())
colors = cm.Reds([0.25 + 0.75 * norm(v) for v in city_res.values])

fig, ax = plt.subplots(figsize=(12, max(5, len(city_res) * 0.45)))

bars = ax.barh(
    city_res.index,
    city_res.values,
    color=colors,
    edgecolor="white",
    linewidth=0.6,
    height=0.7
)

# Annotate values at end of each bar (no overlap)
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + city_res.max() * 0.01,  # slight offset from bar end
        bar.get_y() + bar.get_height() / 2,
        f"{width:.1f}d",
        va="center", ha="left", fontsize=9, color="#333333"
    )

ax.set_title("Average Resolution Time per City", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Avg Resolution Time (days)", fontsize=11)
ax.set_ylabel("City", fontsize=11)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.set_xlim(0, city_res.max() * 1.15)   # extra room for annotations
ax.spines[["top", "right"]].set_visible(False)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

## Agency

In [ ]:
agency_counts = df["Agency"].value_counts()
threshold = agency_counts.sum() * 0.02
main = agency_counts[agency_counts >= threshold]
other = agency_counts[agency_counts <  threshold].sum()

if other > 0:
    main["Other"] = other

palette = [
    "#FF6B6B", "#FF8E53", "#FFC75F", "#F9F871",
    "#A8E6CF", "#3ECFCF", "#4D9DE0", "#7B5EA7",
    "#E84393", "#FF4F81", "#00C9A7", "#845EC2",
    "#F6AE2D", "#2EC4B6", "#E71D36", "#FF9F1C",
    "#6A0572", "#1B998B", "#C5283D", "#E9C46A"
]
colors = palette[:len(main)]

fig, ax = plt.subplots(figsize=(10, 8))

wedges, texts, autotexts = ax.pie(
    main.values,
    labels=None,
    autopct=lambda p: f"{p:.1f}%" if p >= 2 else "",
    colors=colors,
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)

for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight("bold")
    at.set_color("white")

ax.legend(
    wedges,
    [f"{name}  ({val:,})" for name, val in zip(main.index, main.values)],
    title="Agency",
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    fontsize=9,
    title_fontsize=10,
    frameon=True
)

ax.set_title("Complaint Distribution by Agency", fontsize=15, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


<table>
  <thead>
    <tr>
      <th>Agency</th>
      <th>Primary Job</th>
      <th>Common 311 Complaints</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>EDC</td>
      <td>Real estate and economic growth.</td>
      <td>Issues at city-owned maritime or industrial sites.</td>
    </tr>
    <tr>
      <td>TLC</td>
      <td>Regulating taxis, Ubers, and Lyfts.</td>
      <td>Driver harassment, overcharging, lost items in cabs.</td>
    </tr>
    <tr>
      <td>DOE</td>
      <td>Running the city's public school system.</td>
      <td>School maintenance, busing issues.</td>
    </tr>
    <tr>
      <td>DOB</td>
      <td>Regulating construction and structural safety.</td>
      <td>Illegal conversions, crane safety, construction noise.</td>
    </tr>
    <tr>
      <td>OOS</td>
      <td>Climate and sustainability policy.</td>
      <td>Usually high-level policy (rarely seen in 311).</td>
    </tr>
    <tr>
      <td>DPR</td>
      <td>Managing parks, trees, and public green spaces.</td>
      <td>Dead trees, overgrown grass, park maintenance.</td>
    </tr>
    <tr>
      <td>DOHMH</td>
      <td>Public health and restaurant inspections.</td>
      <td>Rats/rodents, food poisoning, smoking violations.</td>
    </tr>
    <tr>
      <td>OTI</td>
      <td>The city's IT department.</td>
      <td>LinkNYC kiosks, public Wi-Fi, city website issues.</td>
    </tr>
    <tr>
      <td>DCWP</td>
      <td>Protecting consumers and workers.</td>
      <td>Consumer scams, price gouging, tow truck issues.</td>
    </tr>
    <tr>
      <td>HPD</td>
      <td>Ensuring residential buildings are safe/livable.</td>
      <td>Heat/Hot Water (very common), mold, lead paint.</td>
    </tr>
    <tr>
      <td>DOT</td>
      <td>Maintaining the "movement" of the city.</td>
      <td>Potholes, broken street lights, traffic signals.</td>
    </tr>
    <tr>
      <td>DHS</td>
      <td>Managing the homeless shelter system.</td>
      <td>Homeless encampments, requests for shelter.</td>
    </tr>
    <tr>
      <td>DSNY</td>
      <td>Trash, recycling, and keeping streets clean.</td>
      <td>Missed trash collection, dirty sidewalks, snow removal.</td>
    </tr>
    <tr>
      <td>DEP</td>
      <td>Managing water, sewers, and air/noise pollution.</td>
      <td>Water leaks, hydrants, sewer backups, air quality.</td>
    </tr>
    <tr>
      <td>NYPD</td>
      <td>Public safety and quality of life.</td>
      <td>Noise, illegal parking, wellness checks.</td>
    </tr>
  </tbody>
</table>

## Location Type

In [ ]:
# Count location type occurrences per city
pivot = (
    df.dropna(subset=["City", "Location Type"])
    .groupby(["City", "Location Type"])
    .size()
    .unstack(fill_value=0)
)
# Top 20 cities by total complaints
top20_cities = pivot.sum(axis=1).nlargest(10).index
pivot = pivot.loc[top20_cities]
# Keep top 4 location types, merge rest into "Other"
top4 = pivot.sum().nlargest(4).index.tolist()
pivot_top4 = pivot[top4].copy()
pivot_top4["Other"] = pivot.drop(columns=top4).sum(axis=1)
# Normalize to %
pivot_pct = pivot_top4.div(pivot_top4.sum(axis=1), axis=0) * 100
# Sort by total complaints descending
city_totals = pivot.sum(axis=1).sort_values(ascending=False)
pivot_pct = pivot_pct.loc[city_totals.index]
colors = ["#4f8ef7", "#e74c3c", "#2ecc71", "#f39c12", "#aaaaaa"]
fig, ax = plt.subplots(figsize=(16, 7))
pivot_pct.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors,
    edgecolor="white",
    linewidth=0.5,
    width=0.7
)
ax.set_title("Top 4 Location Types — Top 10 Cities by Complaints", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("City", fontsize=11)
ax.set_ylabel("Share of Complaints (%)", fontsize=11)
ax.set_ylim(0, 105)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=40, ha="right", fontsize=9)
ax.legend(
    title="Location Type",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=9,
    title_fontsize=10,
    frameon=True
)
plt.tight_layout()
plt.show()

## Complaint Type

In [ ]:
complaint_counts = df["Complaint_Type"].value_counts()

plt.figure(figsize=(14, 6))
complaint_counts.plot(kind="bar", color="steelblue", edgecolor="black")

plt.title("Complaint Type Distribution", fontsize=16)
plt.xlabel("Complaint Type", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
complaint_counts = df["Complaint_Type"].value_counts()

fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    complaint_counts,
    autopct='%1.1f%%',
    startangle=140,
    colors=plt.cm.tab20.colors
)

ax.legend(
    wedges,
    complaint_counts.index,
    title="Complaint Type",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=9
)

plt.title("Complaint Type Distribution", fontsize=16)
plt.tight_layout()
plt.show()



## City

In [ ]:
city_counts = df["City"].value_counts()
# Split into >= 1.5% and "Other"
threshold = 0.015 * city_counts.sum()
main = city_counts[city_counts >= threshold]
# Blue palette for main slices, grey for "Other"
blues = plt.cm.Blues(np.linspace(0.9, 0.4, len(main)))
colors = list(blues) 
fig, ax = plt.subplots(figsize=(10, 10))
wedges, texts, autotexts = ax.pie(
    main.values,
    labels=None,                # no labels on the chart to avoid text collision (use legend instead)
    autopct="%1.1f%%",
    startangle=140,
    pctdistance=0.75,
    colors=colors
)
# Style percentage text
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_color("white")
    autotext.set_fontweight("bold")
# Legend to avoid text collision
ax.legend(
    wedges,
    main.index,
    title="City",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=10
)
ax.set_title("Complaints by City (cities < 1.5% grouped as Other)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Only show cities with enough data points to be meaningful
top_cities = city_missing[city_missing['Total Complaints'] > 50]
top_cities['Missing Rate'].plot(kind='bar', figsize=(14, 5), title='Missing Closed Date Rate by City')
plt.ylabel('Missing Rate')
plt.tight_layout()
plt.show()

## NYC Boundaries

In [ ]:
# Using the borough data we imported in "missing values -> city" section
# Plot the map, color-coded by the borough name
nyc_boundaries.plot(
    figsize=(10, 10), 
    column='boro_name', 
    legend=True, 
    cmap='Set2',
    edgecolor='black'
)

plt.title("NYC Borough Boundaries")
plt.axis('off') # Hides the latitude/longitude axes
plt.show()

## Incident Location

In [ ]:
# Drop rows with no location
geo_df = df.dropna(subset=["Location"], inplace = False).copy()

# Parse "POINT (x y)" string into shapely geometry we use shapely to work with Points like this. 
geo_df["geometry"] = gpd.GeoSeries.from_wkt(geo_df["Location"])

# Create GeoDataFrame (CRS 4326 = standard lat/lon)
geo_df = gpd.GeoDataFrame(geo_df, geometry="geometry", crs="EPSG:4326")

# Plot
geo_df.plot(figsize=(10, 8), markersize=1, color="steelblue", alpha=0.5)
plt.title("Complaint Locations")
plt.show()

In [ ]:
df_clean = df.dropna(subset=["Location"])
gdf = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.GeoSeries.from_wkt(df_clean["Location"]),
    crs="EPSG:4326"
)
gdf["lat"] = gdf.geometry.y
gdf["lon"] = gdf.geometry.x
# Sample 10k random points
gdf_sample = gdf.sample(n=min(10_000, len(gdf)), random_state=42)
# Generate a distinct hex color for each unique complaint type
def complaint_to_color(complaint):
    hash_val = int(hashlib.md5(str(complaint).encode()).hexdigest(), 16)
    r = (hash_val >> 16) & 0xFF
    g = (hash_val >> 8) & 0xFF
    b = hash_val & 0xFF
    return f"#{r:02x}{g:02x}{b:02x}"
complaint_col = "Complaint_Type"  
unique_complaints = gdf_sample[complaint_col].unique()
color_map = {c: complaint_to_color(c) for c in unique_complaints}
m = folium.Map(location=[gdf_sample["lat"].mean(), gdf_sample["lon"].mean()], zoom_start=10)
for _, row in gdf_sample.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color=color_map.get(row[complaint_col], "#888888"),
        fill=True,
        fill_color=color_map.get(row[complaint_col], "#888888"),
        fill_opacity=0.6,
        tooltip=str(row[complaint_col])  # Enable hover to see complaint type
    ).add_to(m)
display(m)


In [ ]:
# Sample 50k random points (set random_state for reproducibility)
sample_size = min(10_000, len(gdf))  # won't crash if data < 50k
gdf_sample = gdf.sample(n=sample_size, random_state=42)

print(f"Plotting {len(gdf_sample)} sampled points")

m = folium.Map(
    location=[gdf_sample["lat"].mean(), gdf_sample["lon"].mean()],
    zoom_start=11,
    tiles="CartoDB dark_matter"
)

HeatMap(data=list(zip(gdf_sample["lat"], gdf_sample["lon"])), radius=7, blur=7).add_to(m)

display(m)


## Created Date

In [ ]:
# Complaints created per day
daily_counts = (
    pd.to_datetime(df["Created Date"])
    .dt.date
    .value_counts()
    .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 6))

ax.bar(daily_counts.index, daily_counts.values, color="#4f8ef7", width=0.8, alpha=0.85)

# Average line
avg = daily_counts.values.mean()
ax.axhline(avg, color="#e74c3c", linewidth=1.8, linestyle="--", label=f"Daily Avg: {avg:,.0f}")
ax.legend(fontsize=11, loc="upper right")

ax.set_title("Number of Complaints Created Per Day", fontsize=16, fontweight="bold", pad=14)
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Number of Complaints", fontsize=12)

ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b %Y"))
plt.xticks(rotation=45, ha="right")

ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


## Day of the week

In [ ]:
# Mean complaints per day-of-week across all weeks
daily_counts = (
    pd.to_datetime(df["Created Date"])
    .dt.date
    .value_counts()
    .reset_index()
)
daily_counts.columns = ["date", "count"]
daily_counts["day_of_week"] = pd.to_datetime(daily_counts["date"]).dt.day_name()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
mean_per_day = (
    daily_counts.groupby("day_of_week")["count"]
    .mean()
    .reindex(day_order)
    .round(1)
)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#4f8ef7" if d not in ("Saturday", "Sunday") else "#f7874f" for d in day_order]
bars = ax.bar(mean_per_day.index, mean_per_day.values, color=colors, width=0.6, alpha=0.88)

# Value labels on top of each bar
for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{bar.get_height():,.0f}",
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

ax.set_title("Average Daily Complaints by Day of Week", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Day of Week", fontsize=12)
ax.set_ylabel("Mean Complaints", fontsize=12)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Legend for weekday vs weekend
weekday_patch = mpatches.Patch(color="#4f8ef7", label="Weekday")
weekend_patch = mpatches.Patch(color="#f7874f", label="Weekend")
ax.legend(handles=[weekday_patch, weekend_patch], fontsize=10)

plt.tight_layout()
plt.show()


## Comaplaints per hour

In [ ]:
# Complaints per hour per day, stacked by top 3 complaint types
dt_series = pd.to_datetime(df["Created Date"])

df_temp = df.assign(
    day_of_week=dt_series.dt.day_name(),
    hour=dt_series.dt.hour
)

# Identify top 3 complaint types globally
top3 = df["Complaint_Type"].value_counts().head(3).index.tolist()
palette = ["#4f8ef7", "#f7874f", "#4fc97f"]  # blue, orange, green

# Group by day + hour + complaint type, keep only top 3 + "Other"
df_temp["Type"] = df_temp["Complaint_Type"].where(df_temp["Complaint_Type"].isin(top3), "Other")
hourly = (
    df_temp.groupby(["day_of_week", "hour", "Type"])
    .size()
    .reset_index(name="count")
)

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
hours = list(range(24))
stack_order = top3 + ["Other"]
stack_colors = palette + ["#cccccc"]

fig, axes = plt.subplots(7, 1, figsize=(16, 26), sharey=True)
fig.suptitle("Complaints Per Hour — By Day of Week\n(stacked by top 3 complaint types)",
             fontsize=17, fontweight="bold", y=1.01)

for ax, day in zip(axes, day_order):
    day_data = hourly[hourly["day_of_week"] == day]
    bottoms = [0] * 24

    for complaint, color in zip(stack_order, stack_colors):
        type_data = (
            day_data[day_data["Type"] == complaint]
            .set_index("hour")["count"]
            .reindex(hours, fill_value=0)
        )
        ax.bar(hours, type_data.values, bottom=bottoms,
               color=color, alpha=0.88, width=0.8, label=complaint)
        bottoms = [b + v for b, v in zip(bottoms, type_data.values)]

    ax.set_ylabel(day, fontsize=10, fontweight="bold", rotation=0, labelpad=90, va="center")
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.set_xlim(-0.5, 23.5)
    ax.set_xticks(hours)
    ax.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45, ha="right", fontsize=8)

axes[-1].set_xlabel("Hour of Day (0 = midnight, 23 = 11pm)", fontsize=11)

# Single shared legend (deduplicated)
handles, labels = axes[0].get_legend_handles_labels()
seen = {}
for h, l in zip(handles, labels):
    if l not in seen:
        seen[l] = h
fig.legend(seen.values(), seen.keys(), loc="upper right", fontsize=10, title="Complaint Type")

plt.tight_layout()
plt.show()


## Correlation Heatmap

In [ ]:
# Build a numeric version of the key columns
le = LabelEncoder()
heatmap_df = pd.DataFrame({
    "City":              le.fit_transform(df["City"].astype(str)),
    "Resolution Time":   df["Resolution Time"],
    "Complaint Type":    le.fit_transform(df["Complaint_Type"].astype(str)),
    "Agency":            le.fit_transform(df["Agency"].astype(str)),
    "Day Opened":        le.fit_transform(df["Day_Opened"].astype(str)),
    "Location Type":     le.fit_transform(df["Location Type"].astype(str)),
    "Hour Created":      pd.to_datetime(df["Created Date"]).dt.hour,
    "Month Created":     pd.to_datetime(df["Created Date"]).dt.month,
    "Status":            le.fit_transform(df["Status"].astype(str)),
}).dropna()
corr_matrix = heatmap_df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 10},
    ax=ax
)
ax.set_title("Feature Correlation Heatmap", fontsize=16, fontweight="bold", pad=16)
plt.xticks(rotation=30, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# Encoding data 


In [ ]:
df.head()

We drop status, dates ( except open date so we can add a new feature "night ratio"), and Location ( since those are not the locations of the zip and we will create them later using json file that is availbale online), encode the days, the agency , and remove the city because we want to cluster using the ZIP and later Street.

In [ ]:
# Dropping columns
ZIP_temp_df = df.drop(["Street Name","City", "Closed Date", "Status", "Location"], axis = 1)
ZIP_temp_df["Created Date"] = pd.to_datetime(ZIP_temp_df['Created Date'])

# Save a copy for Street encoding.
STREET_temp_df = df.drop(["City", "Closed Date", "Status", "Location", "Incident Zip"], axis = 1)
STREET_temp_df["Created Date"] = pd.to_datetime(STREET_temp_df['Created Date'])
ZIP_temp_df.head()

## Encoding ( Zip , Street )

In [ ]:
# Input: working df, the target column for clustering
# Output: new reorganized df

def encoding_func(temp_df, tgt_col):
    # Save a working copy

    converted_df = temp_df.copy()

    # ==================================================================
    # Add the new features of weekend

    # Weekend feature: dayofweek Mon=0 … Sun=6 → Sat=5, Sun=6
    converted_df["_is_weekend"] = converted_df["Created Date"].dt.dayofweek.isin([5, 6])

    # Night feature: complaints opened between 22:00–05:00 (start-end)
    hour = converted_df["Created Date"].dt.hour
    converted_df["_is_night"] = (hour >= 22) | (hour < 5)

    # For each ZIP (or STREET) convert the features into percentage of total complaints 
    temporal = converted_df.groupby(tgt_col).agg( # Summarize rows into one ratio row
        Weekend_Ratio = ("_is_weekend", "mean"),
        NightComplaintRatio = ("_is_night", "mean"),
    )

    # Drop temp columns
    converted_df.drop(columns=["_is_weekend", "_is_night"], inplace=True)


    # ==================================================================
    # Convert each complaint into a column representing the ratio

    # Count complaints per ZIP (or STREET) and Complaint_Type
    complaint_counts = (
        converted_df.groupby([tgt_col, "Complaint_Type"])
        .size()
        .unstack(fill_value=0) # Make complaint type be horizontal in the table, using this we count for each zip the amount of complaints
    )

    # For each ZIP (or STREET) convert the complaints into percentage of total complaints 
    complaint_pct = complaint_counts.div(complaint_counts.sum(axis=1), axis=0)  # Divide each complain by the total complaints  

    # Create columns for each complaint and assign the percentage
    complaint_pct.columns = [col + "_Pct" for col in complaint_pct.columns]

    # There is an ambiguous column called "Other/Misc" that also appears in complaint type so we will rename it here.
    complaint_pct = complaint_pct.rename(columns={"Other/Misc_Pct": "Other/Misc_CType_Pct"})

    # ==================================================================                   
    # Using resolution time to create 1) avg resoultion per zip (or STREET) 2) avg resoultions for top 3 complaints in the whole dataset per zip (or STREET)
    # We will use median instead of average since the resolution time doesn't have a normal distribution
    # Median resolution time per ZIP (or STREET)
    median_resolution = (
        converted_df.groupby(tgt_col)["Resolution Time"]
        .median()
        .rename("Median_Resolution_Time")
    )

    # Find the top 3 complaint types
    top3_types = converted_df["Complaint_Type"].value_counts().head(3).index.tolist()

    resolution_top3_frames = []
    for ctype in top3_types:
        col_name = ("Median_" + ctype + "_Resolution")
        per_zip = (
            converted_df[converted_df["Complaint_Type"] == ctype]
            .groupby(tgt_col)["Resolution Time"]
            .median()
            .rename(col_name)
        )
        resolution_top3_frames.append(per_zip)

    # Combine the 3 series 
    resolution_top3 = (pd.concat(resolution_top3_frames, axis=1))



    # ==================================================================
    # Convert each Location Type into a column representing the ratio

    # Count complaints per ZIP (or STREET) and Location Type
    location_counts = (
        converted_df.groupby([tgt_col, "Location Type"])
        .size()
        .unstack(fill_value=0) # Make location type be horizontal in the table, using this we count for each zip the amount of complaints
    )

    # For each ZIP (or STREET) convert the locations into percentage of total complaints 
    location_pct = location_counts.div(location_counts.sum(axis=1), axis=0)


    # Create columns for each location and assign the percentage
    location_pct.columns = ( location_pct.columns + "_Pct" )

    # There is an ambiguous column called "Other/Misc" that also appears in complaint type so we will rename it here.
    location_pct = location_pct.rename(columns={"Other/Misc_Pct": "Other/Misc_LType_Pct"})

    #  ==================================================================
    # Add feature representing the amount of unique agencies operation in the ZIP

    agency_diversity = (
        converted_df.groupby(tgt_col)["Agency"]
        .nunique() # Number of unique values
        .rename("Agency_Diversity")
    )


    # ==================================================================
    # Add a columns of total complaints per ZIP (or STREET)

    total_complaints = (
        converted_df.groupby(tgt_col)
        .size()
        .rename("Total_Complaints")
    )


    # ==================================================================
    # Finally combining all the features into the df


    temp_df = pd.concat(
        [
            total_complaints,    # Total_Complaints
            temporal,            # Weekend_Ratio, NightComplaintRatio
            complaint_pct,       # 12 complaint columns %
            median_resolution,   # median_Resolution_Time 
            resolution_top3,     # top 3 complaints avg resolution time
            location_pct,        # 7 location type columns %
            agency_diversity,    # Num of unique agencies 
        ],
        axis=1,
    ).reset_index()

    print(f"temp_df shape: {temp_df.shape}")
    print(f"{tgt_col}'s (rows)  : {temp_df.shape[0]}")
    print(f"Features     : {temp_df.shape[1] - 1}") # Excluding Incident Zip (or Street Name)

    return temp_df





### Encoding for Incident Zip clustering

In [ ]:
# Running the function above
ZIP_conv_df = encoding_func(ZIP_temp_df, "Incident Zip").copy()

In [ ]:
ZIP_conv_df.head()

#### Data Verification

In [ ]:
ZIP_conv_df.dtypes

In [ ]:
ZIP_conv_df.isnull().sum()

After running the conversion code we got some missing values in the top 3 complaints resolution columns. This might happen since not every zip has those 3 complaints.

In [ ]:
# Zips with missing values in those columns
ZIP_conv_df[ZIP_conv_df.isna().any(axis=1)]

In [ ]:
# Lets check a specific zip 
df[df["Incident Zip"] == "10045"]

When deciding how to deal with the missing values we thought about 2 approaches:

1) we firstly thought to fill in the mean value of the column. But that can affect the centroid of the clustering algorithm.

2. Dropping the rows of the zip, but we have in total 245 rows and dropping 45 rows is deleting about 20% of our data.

3. Dropping the columns of the top 3 resolution time complaint types, and setteling with just the median resolution time - we chose this option.

In [ ]:
ZIP_final_df = ZIP_conv_df.drop(["Median_Noise_Resolution", "Median_Housing & Building Maintenance_Resolution", "Median_Vehicles & Parking_Resolution"], axis=1)
print(f"final df shape: {ZIP_final_df.shape}")
print(f"temp df shape: {ZIP_conv_df.shape}")
ZIP_final_df.head()

In [ ]:
ZIP_final_df.isnull().sum()

#### Renaming Columns

In [ ]:
# Changing column names
old_col = ZIP_final_df.columns

new_col = []
for column in ZIP_final_df.columns:
    new_col.append(column.replace(" & ", "_and_").replace(" ", "_"))
new_col[3]="Night_Complaint_Pct"
ZIP_final_df.columns = new_col

# Make a copy to download
pre_pca_df_final = ZIP_final_df.copy()

In [ ]:
# Verify that columns are the same after change
verification_df = pd.DataFrame({"old_col" : old_col, "new_col" : new_col})
verification_df

#### Normalization

Total Complaints (Volume): This is a raw count. In a dense urban environment, a highly populated ZIP code might generate 50,000 requests, while a smaller commercial or industrial ZIP code might generate 500.

Median Resolution Time (Duration): This is a continuous metric ( hours ) and is highly vulnerable to outliers. A single bureaucratic glitch where tickets stay open for a long time can severely skew a neighborhood's median.

Agency Diversity (Count): It has a much smaller and tighter numerical range than the other two, but it still isn't a proportion.

For "Total Complaints" and "Median Resolution Time" we will use Log Transform since those two columns are heavily skewed right. Then we will use MinMaxScaler on those 3 columns and leave the rest of the columns same.

In [ ]:
# Get the zip codes
zip_codes = ZIP_final_df["Incident_Zip"]

# Get the features without the zip code column
X_features = ZIP_final_df.drop("Incident_Zip", axis=1).copy()

# ── Step 1: Log-transform the two highly skewed volume/time columns ──────────
# np.log1p handles zeros safely (log(1+x)), keeping values non-negative
skewed_cols = ["Total_Complaints", "Median_Resolution_Time"]
X_features[skewed_cols] = X_features[skewed_cols].apply(np.log1p)


# ── Step 2: Build a ColumnTransformer ────────────────────────────────────────
# MinMaxScaler on the 3 macro columns → squeezes them into [0, 1]
# passthrough on the 24 ratio columns → left completely untouched
macro_cols  = ["Total_Complaints", "Median_Resolution_Time", "Agency_Diversity"]
ratio_cols  = [c for c in X_features.columns if c not in macro_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("macro_scaler", MinMaxScaler(), macro_cols),  # scale macro cols
        ("ratio_pass",   "passthrough",  ratio_cols),  # keep ratio cols as-is
    ],
    remainder="drop",        # safety net: drop anything not explicitly listed
    verbose_feature_names_out=False,  # keeps original column names intact
)


# ── Step 3: Fit and transform ─────────────────────────────────────────────────
X_scaled = preprocessor.fit_transform(X_features)

# Reconstruct a tidy DataFrame with columns in the same order as the transformer
col_order = macro_cols + ratio_cols
ZIP_df_scaled = pd.DataFrame(X_scaled, columns=col_order)

# Restore original column order (macro cols first, then ratios) — optional
ZIP_df_scaled = ZIP_df_scaled[X_features.columns]

# Re-insert the zip code as the first column (used later in the corr heatmap)
ZIP_df_scaled.insert(0, "Incident_Zip", zip_codes.values)

# Display the final, clean dataset ready for clustering / NMF
ZIP_df_scaled.head()

#### Correlation Map 

In [ ]:
# Build a numeric version of the key columns
heatmap_df = ZIP_df_scaled.copy()
corr_matrix = heatmap_df.drop("Incident_Zip", axis = 1).corr()
# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    corr_matrix,
    mask=mask,                          # masks the upper triangle
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 10},
    ax=ax
)
ax.set_title("Feature Correlation Heatmap", fontsize=8, fontweight="bold", pad=16)
plt.xticks(rotation=70, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

### Encoding data for Street name clustering

In [ ]:
# We will use STREET_temp_df
STREET_temp_df.head()

In [ ]:
# Running the function at the start of the Encoding section
STREET_conv_df = encoding_func(STREET_temp_df, "Street Name").copy()

In [ ]:
STREET_conv_df.head()

#### Data Verification

In [ ]:
STREET_conv_df.dtypes

In [ ]:
STREET_conv_df.isnull().sum()

Same as in ZIP encoding we will remove the top 3 complaint columns.

In [ ]:
STREET_final_df = STREET_conv_df.drop(["Median_Noise_Resolution", "Median_Housing & Building Maintenance_Resolution", "Median_Vehicles & Parking_Resolution"], axis=1)
print(f"final df shape: {STREET_final_df.shape}")
print(f"temp df shape: {STREET_conv_df.shape}")
STREET_final_df.head()

In [ ]:
STREET_final_df.isnull().sum()

#### Renaming Columns

In [ ]:
# Changing column names
old_col = STREET_final_df.columns

new_col = []
for column in STREET_final_df.columns:
    new_col.append(column.replace(" & ", "_and_").replace(" ", "_"))
new_col[3]="Night_Complaint_Pct"
STREET_final_df.columns = new_col

# Make a copy to download
pre_pca_df_final = STREET_final_df.copy()

In [ ]:
# Verify that columns are the same after change
verification_df = pd.DataFrame({"old_col" : old_col, "new_col" : new_col})
verification_df

#### Normalization

In [ ]:
# Get the zip codes
street_names = STREET_final_df["Street_Name"]

# Get the features without the zip code column
X_features = STREET_final_df.drop("Street_Name", axis=1)

# Standardize the features for the PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

STREET_df_scaled = pd.DataFrame(X_scaled, columns = X_features.columns) # Will be used later in a corr heatmap
STREET_df_scaled.insert(0, "Street_Name", street_names.values)

# Display the final, clean dataset ready for DBSCAN
STREET_df_scaled.head()

In [ ]:
# Get the zip codes
street_names = STREET_final_df["Street_Name"]

# Get the features without the zip code column
X_features = STREET_final_df.drop("Street_Name", axis=1).copy()

# ── Step 1: Log-transform the two highly skewed volume/time columns ──────────
# np.log1p handles zeros safely (log(1+x)), keeping values non-negative
skewed_cols = ["Total_Complaints", "Median_Resolution_Time"]
X_features[skewed_cols] = X_features[skewed_cols].apply(np.log1p)


# ── Step 2: Build a ColumnTransformer ────────────────────────────────────────
# MinMaxScaler on the 3 macro columns → squeezes them into [0, 1]
# passthrough on the 24 ratio columns → left completely untouched
macro_cols  = ["Total_Complaints", "Median_Resolution_Time", "Agency_Diversity"]
ratio_cols  = [c for c in X_features.columns if c not in macro_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("macro_scaler", MinMaxScaler(), macro_cols),  # scale macro cols
        ("ratio_pass",   "passthrough",  ratio_cols),  # keep ratio cols as-is
    ],
    remainder="drop",        # safety net: drop anything not explicitly listed
    verbose_feature_names_out=False,  # keeps original column names intact
)


# ── Step 3: Fit and transform ─────────────────────────────────────────────────
X_scaled = preprocessor.fit_transform(X_features)

# Reconstruct a tidy DataFrame with columns in the same order as the transformer
col_order = macro_cols + ratio_cols
STREET_df_scaled = pd.DataFrame(X_scaled, columns=col_order)

# Restore original column order (macro cols first, then ratios) — optional
STREET_df_scaled = STREET_df_scaled[X_features.columns]

# Re-insert the zip code as the first column (used later in the corr heatmap)
STREET_df_scaled.insert(0, "Street_Name", street_names.values)

# Display the final, clean dataset ready for clustering / NMF
STREET_df_scaled.head()

#### Correlation Map 

In [ ]:
# Build a numeric version of the key columns
heatmap_df = STREET_df_scaled.copy()
corr_matrix = heatmap_df.drop("Street_Name", axis = 1).corr()
# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
fig, ax = plt.subplots(figsize=(20, 16))
sns.heatmap(
    corr_matrix,
    mask=mask,                          # ← masks the upper triangle
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": 10},
    ax=ax
)
ax.set_title("Feature Correlation Heatmap", fontsize=8, fontweight="bold", pad=16)
plt.xticks(rotation=70, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

#### NMF

In [ ]:
STREET_df_scaled.shape

In [ ]:
# ── Step 1: Isolate the Data ──────────────────────────────────────────────────
# NMF can only take the numbers, so we must drop the ZIP or Street identifier.
X_nmf = STREET_df_scaled.drop("Street_Name", axis=1)

# Calculate the denominator for our per-cell error math
total_cells = X_nmf.shape[0] * X_nmf.shape[1]
sqrt_total_cells = np.sqrt(total_cells)

# ── Step 2: Loop and Calculate ────────────────────────────────────────────────
k_range = range(2, 18)  # Testing from 2 up to 14 components
avg_errors_per_cell = []

print("Running NMF models... this might take a few seconds.")

for k in k_range:
    # init='nndsvda' is the optimal initialization method for dense data in NMF.
    # max_iter=1000 ensures the model has enough time to converge.
    nmf_model = NMF(n_components=k, init='nndsvda', random_state=42, max_iter=1000)
    nmf_model.fit(X_nmf)
    
    # Store the mathematical error (Frobenius norm)
    # Convert raw Frobenius norm to Average Error Per Cell
    raw_error = nmf_model.reconstruction_err_
    per_cell_error = raw_error / sqrt_total_cells
    
    avg_errors_per_cell.append(per_cell_error)

# ── Step 3: Plot the Elbow Curve ──────────────────────────────────────────────
plt.figure(figsize=(10, 6))
plt.plot(k_range, avg_errors_per_cell, marker='o', linestyle='--', linewidth=2)

plt.title('NMF Elbow Method: Finding Optimal Components (k)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Components (k)', fontsize=12)
plt.ylabel('Reconstruction Error (%)', fontsize=12)
plt.xticks(k_range)
plt.grid(True, linestyle=':', alpha=0.7)

# Highlight the plot for easy reading
plt.tight_layout()
plt.show()

We will choose k = 9.

In [ ]:
# 1. Initialize and fit the NMF model
k = 9
nmf_model = NMF(n_components=k, init='nndsvda', random_state=42, max_iter=1000)

# Fit the model and extract the W matrix (the compressed dataset)
W_matrix = nmf_model.fit_transform(X_nmf)

# 2. Extract the H matrix (the recipes) to find what each profile represents
H_matrix = nmf_model.components_
original_features = X_nmf.columns

# 3. Dynamically generate descriptive column names
dynamic_profile_names = []
for i in range(k):
    # Get the feature weights for this specific profile
    profile_weights = H_matrix[i]
    
    # Find the index of the feature with the absolute highest weight
    top_feature_index = profile_weights.argmax()
    top_feature_name = original_features[top_feature_index]
    
    # Create an instantly readable name
    dynamic_profile_names.append(f"Profile_{i+1} ({top_feature_name})")

# 4. Convert the raw numpy array back into a readable Pandas DataFrame
STREET_df_NMF = pd.DataFrame(W_matrix, columns=dynamic_profile_names)

# 5. Bring back the identifier column
STREET_df_NMF.insert(0, "Street_Name", STREET_df_scaled["Street_Name"].values)

# Display the final, self-explaining dataset ready for clustering
STREET_df_NMF.head()

# Downloading Final data

In [ ]:
# Let's save the table as csv
STREET_final_df.to_csv("display_rdy_STREET_4.csv", index=False)
ZIP_final_df.to_csv("display_rdy_ZIP_4.csv", index=False)
STREET_df_scaled.to_csv("norm_STREET_5.csv", index=False)
ZIP_df_scaled.to_csv("norm_ZIP_5.csv", index=False)
streets_projected[["full_street_name", "geometry"]].to_file("nyc_clustered_streets.gpkg", driver="GPKG")
# Dont forget to put them in the Data folder! 